## SkyCoord, Frame and Angles as xarray Dataset

SkyCoord and Frame fully support conversion to datasets preserving:
* Representation
* Differential
* Frame
* Angle (Longitude and Latitude)

Conversion to dataset supports optionally providing axis (xarray coordinate) labels.

In [ ]:
import astropy.units as u
import numpy as np
import xarray as xr
from astropy.coordinates import ICRS, SkyCoord
from astropy.time import Time

from astropy_xarray.coordinates.sky_coord import (
    skycoord_to_dataset,
)

sky_direction = SkyCoord(
    ra=[[2, 6, 7, 4]] * u.deg,
    dec=[[4, 7, 4, 3]] * u.deg,
    pm_ra_cosdec=[[1, 1, 1, 1]] * u.mas / u.yr,
    pm_dec=[[1, 1, 1, 1]] * u.mas / u.yr,
    frame="icrs",
    # NOTE: for explicit direction, consider using:
    # frame=ICRS(
    #     representation_type='unitspherical',
    #     differential_type='unitsphericalcoslat',
    # ),
)
display(sky_direction)
skycoord_to_dataset(
    sky_direction,
    coords={
        "timestamp": ("time", Time([1.7e9], format="unix")),
        "field_label": ("field", ["a", "b", "c", "d"]),
    },
)

SkyCoordinate representation and differential types for both display and data are detected. In other words, radial_velocity when **implicit** is not recorded as a data_var.

* *display* differential_type is `'sphericalcoslat'`, but
* *data* differential_type is `'unitsphericalcoslat'`

In [ ]:
sky_position = SkyCoord(
    ra=[2, 6, 7, 4] * u.deg,
    dec=[4, 7, 4, 3] * u.deg,
    distance=[4, 3, 6, 4] * u.parsec,
    pm_ra_cosdec=[1, 1, 1, 1] * u.mas / u.yr,
    pm_dec=[1, 1, 1, 1] * u.mas / u.yr,
    frame="icrs",
)
display(sky_position)
display(
    skycoord_to_dataset(
        sky_position, coords={"field_label": ("field", ["a", "b", "c", "d"])}
    )
)

In [ ]:
# Since the default ICRS differential type is 'sphericalcoslat', the first access
# to 'radial_velocity' will compute, cache and mutate the skycoordinate data
display(sky_position.radial_velocity)

# skycoord_to_dataset will then subsequently include these radial velocity values when this occurs
display(
    skycoord_to_dataset(
        sky_position, coords={"field_label": ("field", ["a", "b", "c", "d"])}
    )
)

## Convert Dataset to SkyCoord

SkyCoord datasets use a frame attribute for converting to a SkyCoordinate.

In [ ]:
from astropy_xarray.coordinates import dataset_to_skycoord, load_frame

ds = skycoord_to_dataset(
    sky_position, coords={"field_label": ("field", ["a", "b", "c", "d"])}
)

# frame specific info stored as dataset attribute
assert load_frame(ds.attrs["frame"]) == sky_position.replicate_without_data()

# dataset
result = dataset_to_skycoord(ds)
display(result)
np.testing.assert_array_equal(result.ra, sky_position.ra)
np.testing.assert_array_equal(result.dec, sky_position.dec)
np.testing.assert_array_equal(result.distance, sky_position.distance)

In [ ]:
# accessor available
ds.astropy.to_skycoord()

## Serialization and Deserialization

Using dequantifing, all type metadata can be converted entriely to json-like attributes:

In [ ]:
# serializable layout
dds = skycoord_to_dataset(
    sky_direction, coords={"field_label": ("field", [0, 1, 2, 3])}
).astropy.dequantify()
dds

This allows for compatibility with simple serialization implementations (no custom type handling required).

In [ ]:
import msgpack_numpy


def to_msgpack_numpy(ds: xr.Dataset):
    return msgpack_numpy.packb(ds.to_dict(data="array"))


def from_msgpack_numpy(buf: memoryview):
    return xr.Dataset.from_dict(msgpack_numpy.unpackb(buf))


sc = SkyCoord(ra=np.random.random(1000) * u.rad, dec=np.random.random(1000) * u.rad)
ds = skycoord_to_dataset(sc)
display(ds)

raw = to_msgpack_numpy(ds.astropy.dequantify())
display(raw)

result = from_msgpack_numpy(raw).astropy.quantify()
display(result)

## SkyCoordinate Quirks

In [ ]:
# SkyCoord itself doesn't strongly infer directional reprentation and differential types.
import astropy.units as u
from astropy.coordinates import SkyCoord

sky_direction = SkyCoord(
    ra=[2, 6, 7, 4] * u.deg,
    dec=[4, 7, 4, 3] * u.deg,
    pm_ra_cosdec=[1, 1, 1, 1] * u.mas / u.yr,
    pm_dec=[1, 1, 1, 1] * u.mas / u.yr,
    frame="icrs",
)
assert (
    sky_direction.representation_type.name == "spherical"
)  #  would expect "unitspherical"
assert (
    sky_direction.differential_type.name == "sphericalcoslat"
)  # would expect "unitsphericalcoslat"

In [ ]:
# For differential mismatch, this effects SkyCoord methods until the prefered representations are cached.
sky_position = SkyCoord(
    ra=[2, 6, 7, 4] * u.deg,
    dec=[4, 7, 4, 3] * u.deg,
    distance=[1, 1, 1, 1] * u.dimensionless_unscaled,
    pm_ra_cosdec=[1, 1, 1, 1] * u.mas / u.yr,
    pm_dec=[1, 1, 1, 1] * u.mas / u.yr,
    frame="icrs",
)
assert sky_position.representation_type.name == "spherical"
assert sky_position.differential_type.name == "sphericalcoslat"
sky_position_old_str = str(sky_position)

sky_position.represent_as("spherical", "sphericalcoslat")  # cache updated
assert sky_position.representation_type.name == "spherical"
assert sky_position.differential_type.name == "sphericalcoslat"
assert str(sky_position) != sky_position_old_str  # string method changed

In [ ]:
# To avoid mutable cache quirks, use explicit repr and diff types for unit sphericals
sky_position = SkyCoord(
    ra=[2, 6, 7, 4] * u.deg,
    dec=[4, 7, 4, 3] * u.deg,
    distance=[1, 1, 1, 1] * u.dimensionless_unscaled,
    pm_ra_cosdec=[1, 1, 1, 1] * u.mas / u.yr,
    pm_dec=[1, 1, 1, 1] * u.mas / u.yr,
    frame=ICRS(
        representation_type="spherical",
        differential_type="unitsphericalcoslat",
    ),
)
sky_position_old_str = str(sky_position)
sky_position.represent_as("spherical", "sphericalcoslat")
assert str(sky_position) == sky_position_old_str

In [ ]:
# astropy skycoord frame type casting between directional and positional
# representation may alias data
sky_direction = SkyCoord(
    ra=[2, 6, 7, 4] * u.deg,
    dec=[4, 7, 4, 3] * u.deg,
    pm_ra_cosdec=[1, 1, 1, 1] * u.mas / u.yr,
    pm_dec=[1, 1, 1, 1] * u.mas / u.yr,
    frame="icrs",
)
print("SkyCoord:", sky_direction.ra.data[1])

# dataset serialization avoids unit conversions where possible
ds = skycoord_to_dataset(sky_direction, coords={"field": ["a", "b", "c", "d"]})
print("Dataset:", ds.ra.data[1].value)